# Task 3 — moment tensor inversion

mttime does the inversion. `invert.py` writes its control file, bounds
the depth grid around the GeoNet hypocentre, runs the Clinton loop
(drop stations the solution cannot explain, re-invert), and grades the
result with the INGV table. Sources in the module docstring and
`auto_tdmt.cfg` §3.

In [ ]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "docs" else Path.cwd()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
# work in a scratch archive so the real events/ are untouched
os.environ.setdefault("AUTO_TDMT_EVENTS", str(Path.home() / "work" / "proj_tdmt_NZ" / "notebook_runs"))
import config
from config import P            # every tunable, from auto_tdmt.cfg
EVENT = "2026p669681"
print("parameters from", P.source)

In [ ]:
print(json.dumps(P.as_dict()["invert"], indent=2))

## 3.1 Stations from task 2, Green's functions staged

In [ ]:
import greens, invert, waveforms
from geonet import get_event
ev = get_event(EVENT)
band = config.band_candidates(ev.prelim_mag)[0]
wd = config.EVENTS_DIR / ev.public_id / "task3"
pool, dropped = waveforms.fetch_and_process(ev, wd, band)
model = config.model_for_event(ev.latitude, ev.longitude)
depths = invert.search_depths(ev, model)
print(f"{len(pool)} stations; model {model}; depth grid "
      f"{depths[0]:g}-{depths[-1]:g} km ({len(depths)} depths) around GeoNet {ev.depth_km:g} km")
greens.stage_event_greens(model, pool, depths, band, wd / "greens")

## 3.2 The Clinton loop

In [ ]:
os.chdir(wd)
inv, used, rejected, rounds = invert.clinton_loop(ev, pool, depths, wd, wd / "greens")
os.chdir(ROOT)
print(json.dumps(rounds, indent=1))
for r in rejected:
    print(f"dropped {r['station']}: {r['reason']}")

## 3.3 The solution, its depth curve and its grade

In [ ]:
sol = invert.summarize(inv, ev, used, dropped + rejected, model, rounds)
os.chdir(wd)
sol["jackknife"] = invert.jackknife(ev, used, sol["preferred"]["depth_km"], wd, wd / "greens", sol["preferred"]["plane1"])
os.chdir(ROOT)
sol["quality"] = invert.quality_gates(sol)
p, q = sol["preferred"], sol["quality"]
print(f"Mw {p['mw']:.2f}  depth {p['depth_km']:g} km [{q['depth_range_km'][0]:g}-{q['depth_range_km'][1]:g}]  "
      f"VR {p['vr']:.1f}  DC {p['pdc']:.0f}  grade {q['grade']}  ({q['n_stations_used']} stations)")
print("plane1", p["plane1"]); print("plane2", p["plane2"])
print("checks", q["checks"]); print("jackknife", {k: v for k, v in sol["jackknife"].items() if k != "subsets"})
import matplotlib.pyplot as plt
d = sorted(sol["depth_search"], key=lambda r: r["depth_km"])
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot([r["depth_km"] for r in d], [r["vr"] for r in d], "k.-", label="VR")
ax.axvspan(*q["depth_range_km"], color="0.85", label=f"within {P.invert.depthUncPct:g}% of max")
ax.axvline(ev.depth_km, color="#D55E00", ls="--", label="GeoNet")
ax2 = ax.twinx(); ax2.plot([r["depth_km"] for r in d], [r["pdc"] for r in d], "-", color="#0072B2", alpha=0.6, label="%DC")
ax.set_xlabel("depth (km)"); ax.set_ylabel("VR (%)"); ax2.set_ylabel("%DC", color="#0072B2")
ax.legend(loc="lower right", fontsize=8); plt.tight_layout(); plt.show()

## 3.4 Per-station fit

In [ ]:
print(f"{'station':14s} {'dist':>5s} {'az':>4s} {'own VR':>7s} {'shift s':>8s}")
for r in used:
    print(f"{invert.station_id(r):14s} {r['distance_km']:5.0f} {r['azimuth']:4.0f} {r['station_vr']:7.1f} {r['zcor_s']:+8.1f}")

## What to check
- Do the dropped stations deserve it? (`invert.stationVRDrop`, `stationVRFloor`, `maxTimeShiftS`)
- Is the depth curve a plateau or a spike? Does the shaded range include GeoNet's depth?
  (`invert.depthWindowKm`, `depthUncPct`)
- Would you publish this? Compare your call with the grade (`invert.gradeA_VR` ... `dcMinPublish`).